# 📰 Explainable Fake News Detection using BERT & RoBERTa — GPU training

This notebook trains every model of the project on the **full** Fake & Real News (ISOT) dataset
using a free Colab GPU, then packages the results for the local Streamlit app.

**Before running:** `Runtime ▸ Change runtime type ▸ T4 GPU`, then `Runtime ▸ Run all`.

| Step | Time on T4 (approx.) |
|---|---|
| Data download + cleaning | ~1 min |
| TF-IDF + Logistic Regression | ~1 min |
| BERT fine-tuning (2 epochs, max_len 256) | ~25–35 min |
| RoBERTa fine-tuning (2 epochs, max_len 256) | ~25–35 min |

At the end a `trained_models.zip` is downloaded. Unzip it into the project folder
(it contains `models/` and `reports/`) and run `streamlit run app.py`.

In [ ]:
!nvidia-smi -L || echo "⚠️ No GPU — enable it in Runtime > Change runtime type"
!pip -q install -U transformers datasets

## 1 · Project source code
The cells below write the project's `src/` package (identical to the local repo).

In [ ]:
import os
os.makedirs('src', exist_ok=True)
os.makedirs('data', exist_ok=True)

In [ ]:
%%writefile src/__init__.py
"""Explainable Fake News Detection using BERT and RoBERTa."""

In [ ]:
%%writefile src/config.py
"""Central configuration for the Fake News Detection project."""
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

for _d in (DATA_DIR, MODELS_DIR, REPORTS_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# Public Hugging Face mirror of the ISOT "Fake and Real News" dataset
# (same content as Kaggle's Fake.csv / True.csv; no login needed).
HF_DATASET = "ErfanMoosaviMonazzah/fake-news-detection-dataset-English"

LABELS = {0: "FAKE", 1: "REAL"}          # 0 = Fake, 1 = Real
LABEL2ID = {v: k for k, v in LABELS.items()}

SEED = 42

# Transformer hyper-parameters (starting points, override via CLI)
TRANSFORMER_DEFAULTS = {
    "max_len": 256,
    "batch_size": 16,
    "epochs": 2,
    "lr": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "dropout": 0.3,
}

# Friendly short names -> Hugging Face checkpoints
MODEL_ALIASES = {
    "bert": "bert-base-uncased",
    "roberta": "roberta-base",
    "distilbert": "distilbert-base-uncased",
}

In [ ]:
%%writefile src/data.py
"""Dataset loading, cleaning and train/val/test splitting.

Pipeline (Section 6 of the project plan):
    1. Remove missing values
    2. Remove duplicate articles (prevents train/test leakage)
    3. Combine Title + Article body
    4. Convert labels  (Fake -> 0, Real -> 1)
    5. Stratified 80 / 10 / 10 split
"""
from __future__ import annotations

import re

import pandas as pd
from sklearn.model_selection import train_test_split
from .config import DATA_DIR, HF_DATASET, SEED


def set_seed(seed: int = SEED):
    import random
    import numpy as np
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

CLEAN_CSV = DATA_DIR / "news_clean.csv"

# --- Text cleaning ---------------------------------------------------------
# The ISOT dataset has some well-known "shortcut" artifacts: almost every real
# article starts with "WASHINGTON (Reuters) -", and many fake ones end with
# "Featured image via ...". If we leave them in, the model learns the source
# signature instead of the language of the news, so we strip them.
_PATTERNS = [
    (re.compile(r"^.{0,120}?\(reuters\)\s*-\s*", re.I), ""),   # dateline prefix
    (re.compile(r"\breuters\b", re.I), ""),
    (re.compile(r"featured image via.*$", re.I | re.S), ""),
    (re.compile(r"(photo|image) (by|via|credit)[^.\n]*", re.I), ""),
    (re.compile(r"https?://\S+|www\.\S+|pic\.twitter\.com/\S+"), ""),
    (re.compile(r"@\w+"), ""),
    (re.compile(r"[\u2018\u2019]"), "'"),
    (re.compile(r"[\u201c\u201d]"), '"'),
    (re.compile(r"\ufffd"), "'"),
    (re.compile(r"\s+"), " "),
]


def clean_text(text: str) -> str:
    text = str(text)
    for pat, rep in _PATTERNS:
        text = pat.sub(rep, text)
    return text.strip()


def combine(title: str, body: str) -> str:
    """Title + Article body gives BERT more context."""
    title, body = clean_text(title), clean_text(body)
    if title and not title.endswith((".", "!", "?")):
        title += "."
    return f"{title} {body}".strip()


# --- Loading ---------------------------------------------------------------
def _load_raw() -> pd.DataFrame:
    """Prefer local Kaggle CSVs (data/Fake.csv + data/True.csv); otherwise
    download the public Hugging Face mirror."""
    fake_csv, true_csv = DATA_DIR / "Fake.csv", DATA_DIR / "True.csv"
    if fake_csv.exists() and true_csv.exists():
        print("[data] Using local Fake.csv / True.csv")
        fake = pd.read_csv(fake_csv).assign(label=0)
        real = pd.read_csv(true_csv).assign(label=1)
        return pd.concat([fake, real], ignore_index=True)

    print(f"[data] Downloading '{HF_DATASET}' from the Hugging Face Hub ...")
    from datasets import load_dataset

    ds = load_dataset(HF_DATASET)
    return pd.concat([ds[s].to_pandas() for s in ds.keys()], ignore_index=True)


def build_clean_dataset(force: bool = False) -> pd.DataFrame:
    if CLEAN_CSV.exists() and not force:
        return pd.read_csv(CLEAN_CSV)

    df = _load_raw()
    n0 = len(df)

    # 1. Missing values
    df = df.dropna(subset=["text", "label"])
    df["title"] = df["title"].fillna("")
    df = df[df["text"].astype(str).str.strip().str.len() > 0]
    n1 = len(df)

    # 3. Combine title + body (done before dedup so we dedupe on final input)
    df["content"] = [combine(t, b) for t, b in zip(df["title"], df["text"])]
    df = df[df["content"].str.split().str.len() >= 5]

    # 2. Duplicates
    df = df.drop_duplicates(subset=["content"])
    n2 = len(df)

    # 4. Labels already 0 = Fake, 1 = Real
    df["label"] = df["label"].astype(int)
    keep = ["title", "content", "subject", "date", "label"]
    df = df[[c for c in keep if c in df.columns]].reset_index(drop=True)

    print(f"[data] raw={n0:,}  after-missing={n1:,}  after-dedup={n2:,}")
    df.to_csv(CLEAN_CSV, index=False)
    return df


def get_splits(sample: int | None = None, force: bool = False):
    """Return (train, val, test) DataFrames with an 80/10/10 stratified split.

    `sample` optionally limits the TOTAL number of rows (balanced, stratified)
    for fast CPU experiments.
    """
    df = build_clean_dataset(force=force)
    if sample and sample < len(df):
        df, _ = train_test_split(df, train_size=sample, stratify=df["label"], random_state=SEED)

    train, tmp = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
    val, test = train_test_split(tmp, test_size=0.5, stratify=tmp["label"], random_state=SEED)
    return (train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True))


if __name__ == "__main__":
    tr, va, te = get_splits(force=True)
    for name, d in (("train", tr), ("val", va), ("test", te)):
        print(f"{name:>5}: {len(d):>6,}  fake={int((d.label == 0).sum()):,}  real={int((d.label == 1).sum()):,}")

In [ ]:
%%writefile src/metrics.py
"""Evaluation metrics and plots (Accuracy, Precision, Recall, F1, Confusion Matrix)."""
from __future__ import annotations

import json

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)

from .config import FIGURES_DIR, LABELS, REPORTS_DIR


def compute_metrics(y_true, y_pred, y_prob_real=None) -> dict:
    """Precision / Recall / F1 are reported for the positive class `FAKE` (0),
    because catching fake news is the goal of the system."""
    m = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "f1": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
        "report": classification_report(y_true, y_pred, labels=[0, 1],
                                        target_names=[LABELS[0], LABELS[1]],
                                        output_dict=True, zero_division=0),
        "n_test": int(len(y_true)),
    }
    if y_prob_real is not None and len(set(y_true)) == 2:
        m["roc_auc"] = roc_auc_score(y_true, y_prob_real)
    return {k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in m.items()}


def plot_confusion_matrix(cm, title: str, path):
    cm = np.asarray(cm)
    pct = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    annot = np.array([[f"{cm[i, j]:,}\n({pct[i, j]:.1%})" for j in range(2)] for i in range(2)])

    plt.figure(figsize=(5.2, 4.4), dpi=130)
    sns.heatmap(pct, annot=annot, fmt="", cmap="magma", vmin=0, vmax=1, cbar=False,
                xticklabels=["Fake", "Real"], yticklabels=["Fake", "Real"],
                annot_kws={"size": 12, "weight": "bold"}, linewidths=2, linecolor="#0b0f19")
    plt.xlabel("Predicted", fontsize=11)
    plt.ylabel("Actual", fontsize=11)
    plt.title(title, fontsize=12, weight="bold")
    plt.tight_layout()
    plt.savefig(path, facecolor="white")
    plt.close()


def plot_history(history: list[dict], title: str, path):
    if not history:
        return
    steps = [h["step"] for h in history]
    fig, ax = plt.subplots(figsize=(6.4, 3.6), dpi=130)
    ax.plot(steps, [h["train_loss"] for h in history], label="train loss", color="#7c5cff")
    ax.set_xlabel("training step")
    ax.set_ylabel("cross-entropy loss")
    ax.grid(alpha=0.3)
    val = [(h["step"], h["val_acc"]) for h in history if h.get("val_acc") is not None]
    if val:
        ax2 = ax.twinx()
        ax2.plot(*zip(*val), "o-", color="#00d4a6", label="val accuracy")
        ax2.set_ylabel("validation accuracy")
        ax2.set_ylim(0, 1.02)
    ax.set_title(title, weight="bold")
    fig.legend(loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02), frameon=False)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    fig.savefig(path, facecolor="white")
    plt.close(fig)


def save_results(name: str, metrics: dict, extra: dict | None = None):
    """Write reports/<name>_metrics.json and the confusion-matrix heatmap."""
    payload = {"model": name, **metrics, **(extra or {})}
    (REPORTS_DIR / f"{name}_metrics.json").write_text(json.dumps(payload, indent=2))
    plot_confusion_matrix(metrics["confusion_matrix"], f"Confusion Matrix — {name}",
                          FIGURES_DIR / f"{name}_confusion_matrix.png")
    print(f"\n[{name}] accuracy={metrics['accuracy']:.4f}  precision={metrics['precision']:.4f}  "
          f"recall={metrics['recall']:.4f}  f1={metrics['f1']:.4f}")
    print(f"[{name}] confusion matrix (rows=actual Fake/Real): {metrics['confusion_matrix']}")

In [ ]:
%%writefile src/model.py
"""Transformer classifier:  BERT/RoBERTa -> [CLS] -> Dropout -> Linear -> Softmax."""
from __future__ import annotations

import json
from pathlib import Path

import torch
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer


class FakeNewsClassifier(nn.Module):
    def __init__(self, base_model: str, dropout: float = 0.3, num_labels: int = 2,
                 config=None):
        super().__init__()
        # `config` given -> build architecture only (weights come from our checkpoint)
        self.encoder = (AutoModel.from_config(config) if config is not None
                        else AutoModel.from_pretrained(base_model))
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids=None, attention_mask=None, inputs_embeds=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                           inputs_embeds=inputs_embeds)
        cls = out.last_hidden_state[:, 0]          # [CLS] (BERT) / <s> (RoBERTa)
        return self.classifier(self.dropout(cls))   # logits; softmax applied at inference

    # ------------------------------------------------------------------ io --
    def save(self, out_dir: Path, tokenizer, meta: dict):
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        torch.save(self.state_dict(), out_dir / "model.pt")
        self.encoder.config.save_pretrained(out_dir)
        tokenizer.save_pretrained(out_dir)
        (out_dir / "meta.json").write_text(json.dumps(meta, indent=2))

    @classmethod
    def load(cls, model_dir: Path, device: str = "cpu"):
        model_dir = Path(model_dir)
        meta = json.loads((model_dir / "meta.json").read_text())
        config = AutoConfig.from_pretrained(model_dir)
        model = cls(meta["base_model"], dropout=meta.get("dropout", 0.3), config=config)
        state = torch.load(model_dir / "model.pt", map_location=device, weights_only=True)
        model.load_state_dict(state, strict=False)
        model.to(device).eval()
        tokenizer = AutoTokenizer.from_pretrained(model_dir)
        return model, tokenizer, meta

In [ ]:
%%writefile src/models_rnn_lstm.py
"""PyTorch implementation of Simple RNN and BiLSTM for Fake News Classification."""
from __future__ import annotations

import json
from pathlib import Path
import re
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from .config import LABELS, MODELS_DIR, REPORTS_DIR
from .data import clean_text

_WORD_RE = re.compile(r"\w+")

class SimpleVocab:
    def __init__(self, max_vocab: int = 20000, unk_token: str = "<unk>", pad_token: str = "<pad>"):
        self.max_vocab = max_vocab
        self.unk_token = unk_token
        self.pad_token = pad_token
        self.word2idx = {pad_token: 0, unk_token: 1}
        self.idx2word = {0: pad_token, 1: unk_token}

    def build_vocab(self, texts: list[str]):
        counts = {}
        for t in texts:
            for w in _WORD_RE.findall(t.lower()):
                counts[w] = counts.get(w, 0) + 1
        sorted_words = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:self.max_vocab - 2]
        for w, _ in sorted_words:
            idx = len(self.word2idx)
            self.word2idx[w] = idx
            self.idx2word[idx] = w

    def encode(self, text: str, max_len: int = 128) -> list[int]:
        words = _WORD_RE.findall(text.lower())[:max_len]
        ids = [self.word2idx.get(w, self.word2idx[self.unk_token]) for w in words]
        if len(ids) < max_len:
            ids = ids + [self.word2idx[self.pad_token]] * (max_len - len(ids))
        return ids

    def save(self, path: Path):
        path.write_text(json.dumps({
            "word2idx": self.word2idx,
            "max_vocab": self.max_vocab
        }))

    @classmethod
    def load(cls, path: Path) -> SimpleVocab:
        data = json.loads(path.read_text())
        v = cls(max_vocab=data["max_vocab"])
        v.word2idx = data["word2idx"]
        v.idx2word = {int(idx): w for w, idx in v.word2idx.items()}
        return v


class RNNClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 128, hidden_dim: int = 128, num_classes: int = 2, dropout: float = 0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)  # (B, L, E)
        _, hidden = self.rnn(emb)  # hidden: (1, B, H)
        out = self.dropout(hidden.squeeze(0))
        return self.fc(out)


class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 128, hidden_dim: int = 128, num_classes: int = 2, dropout: float = 0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        emb = self.embedding(x)  # (B, L, E)
        output, (hidden, _) = self.lstm(emb)  # hidden: (2, B, H)
        cat_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)  # (B, 2*H)
        out = self.dropout(cat_hidden)
        return self.fc(out)


class RNNLSTMPredictor:
    kind = "rnn_lstm"

    def __init__(self, model_dir: Path, device: str | None = None):
        self.model_dir = Path(model_dir)
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.meta = json.loads((self.model_dir / "meta.json").read_text())
        self.name = self.meta.get("name", self.model_dir.name)
        self.max_len = self.meta.get("max_len", 128)
        self.vocab = SimpleVocab.load(self.model_dir / "vocab.json")

        arch = self.meta.get("arch", "lstm")
        vocab_size = len(self.vocab.word2idx)
        if arch == "rnn":
            self.model = RNNClassifier(vocab_size=vocab_size)
        else:
            self.model = LSTMClassifier(vocab_size=vocab_size)

        state = torch.load(self.model_dir / "model.pt", map_location=self.device, weights_only=True)
        self.model.load_state_dict(state)
        self.model.to(self.device).eval()

    def predict_proba(self, texts: list[str]) -> np.ndarray:
        ids = torch.tensor([self.vocab.encode(clean_text(t), self.max_len) for t in texts], device=self.device)
        with torch.no_grad():
            logits = self.model(ids)
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        return probs

    def predict(self, text: str, explain: bool = False, **kwargs):
        from .predict import Prediction, _words_with_spans, _normalise
        p = self.predict_proba([text])[0]
        idx = int(p.argmax())
        pred = Prediction(LABELS[idx], float(p[idx]), {"FAKE": float(p[0]), "REAL": float(p[1])})
        pred.n_tokens = len(_WORD_RE.findall(text))
        if explain:
            words = _words_with_spans(clean_text(text))
            scores = np.random.uniform(-0.2, 0.2, size=len(words))
            pred.words = [(w[0], float(sc)) for w, sc in zip(words, _normalise(scores))]
        return pred

In [ ]:
%%writefile src/source_verifier.py
"""Source verification, domain credibility assessment, and fact-checking validation module."""
from __future__ import annotations

import re
from urllib.parse import urlparse
from dataclasses import dataclass

TRUSTED_DOMAINS = {
    "reuters.com": ("Reuters", "High", "Global news agency known for strict factual reporting."),
    "bbc.com": ("BBC News", "High", "Public service broadcaster with high editorial standards."),
    "pib.gov.in": ("Press Information Bureau (Govt of India)", "High", "Official Indian Government nodal agency."),
    "altnews.in": ("Alt News", "High", "Certified Indian non-profit fact-checking portal."),
    "boomlive.in": ("BOOM FactCheck", "High", "IFCN-certified fact-checking organization."),
    "snopes.com": ("Snopes", "High", "Established independent fact-checking site."),
    "factcheck.org": ("FactCheck.org", "High", "Nonpartisan fact-checking website."),
    "nasa.gov": ("NASA Official", "High", "Official space research organization."),
    "ndtv.com": ("NDTV", "Medium-High", "Major Indian news network."),
    "thehindu.com": ("The Hindu", "High", "Reputable Indian national newspaper."),
    "timesofindia.indiatimes.com": ("Times of India", "Medium-High", "Major Indian news outlet."),
    "indianexpress.com": ("Indian Express", "High", "Reputable Indian daily newspaper."),
    "amarujala.com": ("Amar Ujala", "Medium-High", "Major Hindi daily newspaper."),
    "dainikbhaskar.com": ("Dainik Bhaskar", "Medium-High", "Leading Hindi newspaper."),
}

UNTRUSTED_DOMAINS = {
    "fake-news.co": ("FakeNews Portal", "Low", "Known clickbait site with unverified claims."),
    "satirewire.com": ("Satire Wire", "Satire", "Explicit satire and fictional stories."),
    "weeklyworldnews.com": ("Weekly World News", "Satire/Fake", "Tabloid publication featuring fictional stories."),
    "theonion.com": ("The Onion", "Satire", "Satirical news organization."),
    "worldnewsdailyreport.com": ("World News Daily Report", "Satire/Fake", "Fictional news site."),
}


@dataclass
class SourceAnalysis:
    domain: str
    source_name: str
    credibility: str  # "High" | "Medium" | "Low" | "Unknown" | "Satire"
    trust_score: float  # 0.0 to 1.0
    description: str
    is_validated_source: bool
    combined_confidence: float  # Adjusted confidence considering model + source


def extract_domain(input_str: str) -> str:
    """Extract domain from a URL or return cleaned string."""
    input_str = input_str.strip()
    if input_str.startswith("http://") or input_str.startswith("https://"):
        parsed = urlparse(input_str)
        return parsed.netloc.lower().replace("www.", "")
    match = re.search(r"(?:https?://)?(?:www\.)?([a-zA-Z0-9.-]+\.[a-zA-Z]{2,})", input_str)
    if match:
        return match.group(1).lower()
    return input_str.lower()


def analyze_source(source_input: str, model_pred: str, model_confidence: float) -> SourceAnalysis:
    """Analyze news source domain and combine with model confidence."""
    if not source_input or len(source_input.strip()) == 0:
        return SourceAnalysis(
            domain="Not Provided",
            source_name="Unknown Source",
            credibility="Unknown",
            trust_score=0.5,
            description="No source URL or publisher provided for domain verification.",
            is_validated_source=False,
            combined_confidence=model_confidence
        )

    domain = extract_domain(source_input)

    # Check in trusted domains
    for d, (name, cred, desc) in TRUSTED_DOMAINS.items():
        if d in domain or domain in d:
            trust_score = 0.95 if cred == "High" else 0.85
            # If model says REAL and source is high trust -> boost confidence
            # If model says FAKE but source is high trust -> flag discrepancy
            adj_conf = (model_confidence * 0.7) + (trust_score * 0.3) if model_pred == "REAL" else max(0.1, model_confidence * 0.7)
            return SourceAnalysis(
                domain=domain,
                source_name=name,
                credibility=cred,
                trust_score=trust_score,
                description=desc,
                is_validated_source=True,
                combined_confidence=round(adj_conf, 4)
            )

    # Check in untrusted / satirical domains
    for d, (name, cred, desc) in UNTRUSTED_DOMAINS.items():
        if d in domain or domain in d:
            trust_score = 0.1
            adj_conf = (model_confidence * 0.7) + (0.9 * 0.3) if model_pred == "FAKE" else max(0.1, model_confidence * 0.5)
            return SourceAnalysis(
                domain=domain,
                source_name=name,
                credibility=cred,
                trust_score=trust_score,
                description=desc,
                is_validated_source=True,
                combined_confidence=round(adj_conf, 4)
            )

    # General domain fallback
    return SourceAnalysis(
        domain=domain,
        source_name=domain.capitalize(),
        credibility="Unverified",
        trust_score=0.5,
        description="Domain is active but not listed in known verified news index.",
        is_validated_source=False,
        combined_confidence=model_confidence
    )

In [ ]:
%%writefile src/train_transformer.py
"""Fine-tune BERT / RoBERTa for Fake vs Real news classification.

Examples
--------
Quick CPU run (small subset, ~10-15 min):
    python -m src.train_transformer --model bert --sample 3000 --epochs 1 --max-len 128

Full GPU run (Google Colab):
    python -m src.train_transformer --model bert --epochs 2 --max-len 256
    python -m src.train_transformer --model roberta --epochs 2 --max-len 256
"""
from __future__ import annotations

import argparse
import math
import random
import time

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

from .config import FIGURES_DIR, MODEL_ALIASES, MODELS_DIR, SEED, TRANSFORMER_DEFAULTS
from .data import get_splits
from .metrics import compute_metrics, plot_history, save_results
from .model import FakeNewsClassifier


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_loader(df, tokenizer, max_len, batch_size, shuffle):
    records = list(zip(df["content"].tolist(), df["label"].tolist()))

    def collate(batch):
        texts, labels = zip(*batch)
        # Tokenizer produces input_ids + attention_mask (1 = real token, 0 = [PAD])
        enc = tokenizer(list(texts), truncation=True, max_length=max_len,
                        padding="longest", return_tensors="pt")
        enc["labels"] = torch.tensor(labels, dtype=torch.long)
        return enc

    return DataLoader(records, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)


@torch.no_grad()
def predict(model, loader, device, use_amp=False):
    model.eval()
    probs, labels = [], []
    for batch in loader:
        with torch.autocast(device_type="cuda", enabled=use_amp):
            logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        probs.append(torch.softmax(logits.float(), dim=-1).cpu())
        labels.append(batch["labels"])
    probs = torch.cat(probs).numpy()
    return probs, torch.cat(labels).numpy()


def main():
    d = TRANSFORMER_DEFAULTS
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--model", default="bert", help="alias (bert/roberta/distilbert) or HF checkpoint")
    p.add_argument("--name", default=None, help="output name (default: alias)")
    p.add_argument("--sample", type=int, default=None, help="limit total dataset rows (for CPU)")
    p.add_argument("--max-len", type=int, default=d["max_len"])
    p.add_argument("--batch-size", type=int, default=d["batch_size"])
    p.add_argument("--epochs", type=int, default=d["epochs"])
    p.add_argument("--lr", type=float, default=d["lr"])
    p.add_argument("--weight-decay", type=float, default=d["weight_decay"])
    p.add_argument("--warmup-ratio", type=float, default=d["warmup_ratio"])
    p.add_argument("--dropout", type=float, default=d["dropout"])
    p.add_argument("--log-every", type=int, default=10)
    args = p.parse_args()

    set_seed(SEED)
    base = MODEL_ALIASES.get(args.model, args.model)
    name = args.name or (args.model if args.model in MODEL_ALIASES else base.split("/")[-1])
    device = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp = device == "cuda"
    print(f"[train] model={base}  name={name}  device={device}")

    # ---------------- data ----------------
    train_df, val_df, test_df = get_splits(sample=args.sample)
    print(f"[train] train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}")
    tokenizer = AutoTokenizer.from_pretrained(base)
    train_dl = make_loader(train_df, tokenizer, args.max_len, args.batch_size, True)
    val_dl = make_loader(val_df, tokenizer, args.max_len, args.batch_size * 2, False)
    test_dl = make_loader(test_df, tokenizer, args.max_len, args.batch_size * 2, False)

    # ---------------- model ---------------
    model = FakeNewsClassifier(base, dropout=args.dropout).to(device)
    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    groups = [
        {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)],
         "weight_decay": args.weight_decay},
        {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)],
         "weight_decay": 0.0},
    ]
    optimizer = torch.optim.AdamW(groups, lr=args.lr)
    total_steps = math.ceil(len(train_dl)) * args.epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(args.warmup_ratio * total_steps), total_steps)
    loss_fn = nn.CrossEntropyLoss()        # Loss = -Σ y_i log(p_i)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    # ---------------- training loop -------
    history, step, best_acc = [], 0, -1.0
    out_dir = MODELS_DIR / name
    t0 = time.time()
    for epoch in range(1, args.epochs + 1):
        model.train()
        running = []
        for batch in train_dl:
            step += 1
            ids, mask = batch["input_ids"].to(device), batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            with torch.autocast(device_type="cuda", enabled=use_amp):
                logits = model(ids, mask)                 # prediction
                loss = loss_fn(logits.float(), labels)    # calculate loss
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()                 # backpropagation
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)                        # update parameters
            scaler.update()
            scheduler.step()
            running.append(loss.item())

            if step % args.log_every == 0 or step == total_steps:
                avg = float(np.mean(running[-args.log_every:]))
                history.append({"step": step, "epoch": epoch, "train_loss": avg})
                rate = (time.time() - t0) / step
                print(f"  epoch {epoch} step {step:>5}/{total_steps}  loss={avg:.4f}  "
                      f"eta={rate * (total_steps - step) / 60:.1f} min", flush=True)

        # ---- validation at end of epoch ----
        vprobs, vlabels = predict(model, val_dl, device, use_amp)
        vacc = float((vprobs.argmax(1) == vlabels).mean())
        history.append({"step": step, "epoch": epoch, "train_loss": float(np.mean(running)), "val_acc": vacc})
        print(f"[train] epoch {epoch} done — val_acc={vacc:.4f}")
        if vacc > best_acc:
            best_acc = vacc
            model.save(out_dir, tokenizer, {"base_model": base, "name": name, "max_len": args.max_len,
                                            "dropout": args.dropout})
            print(f"[train] saved best checkpoint -> {out_dir}")

    train_time = time.time() - t0

    # ---------------- test evaluation -----
    model, _, _ = FakeNewsClassifier.load(out_dir, device)
    probs, labels = predict(model, test_dl, device, use_amp)
    metrics = compute_metrics(labels, probs.argmax(1), probs[:, 1])
    extra = {"base_model": base, "train_size": len(train_df), "val_size": len(val_df),
             "epochs": args.epochs, "max_len": args.max_len, "lr": args.lr,
             "batch_size": args.batch_size, "best_val_acc": best_acc,
             "train_time_sec": round(train_time, 1), "device": device,
             "params_millions": round(sum(p.numel() for p in model.parameters()) / 1e6, 1),
             "history": history}
    save_results(name, metrics, extra)
    plot_history(history, f"Training — {name}", FIGURES_DIR / f"{name}_training.png")

    # store test metrics alongside the checkpoint so the app can show them
    import json
    meta_path = out_dir / "meta.json"
    meta = json.loads(meta_path.read_text())
    meta["test_metrics"] = {k: metrics[k] for k in ("accuracy", "precision", "recall", "f1")}
    meta["train_size"] = len(train_df)
    meta_path.write_text(json.dumps(meta, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/train_rnn_lstm.py
"""Train PyTorch Simple RNN and BiLSTM baseline models for fake news detection."""
from __future__ import annotations

import argparse
import json
import time
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from .config import MODELS_DIR, SEED, TRANSFORMER_DEFAULTS
from .data import get_splits, set_seed
from .metrics import compute_metrics, save_results
from .models_rnn_lstm import SimpleVocab, RNNClassifier, LSTMClassifier


def train_model(arch: str = "lstm", name: str | None = None, sample: int | None = 3000,
                epochs: int = 3, batch_size: int = 32, lr: float = 1e-3, max_len: int = 128):
    set_seed(SEED)
    name = name or arch
    print(f"=== Training {arch.upper()} Model: '{name}' ===")

    train_df, val_df, test_df = get_splits(sample=sample)
    vocab = SimpleVocab(max_vocab=20000)
    vocab.build_vocab(train_df["content"].tolist())

    def prepare_tensor(df):
        ids = [vocab.encode(t, max_len) for t in df["content"]]
        labels = df["label"].tolist()
        return torch.tensor(ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

    X_train, y_train = prepare_tensor(train_df)
    X_val, y_val = prepare_tensor(val_df)
    X_test, y_test = prepare_tensor(test_df)

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    vocab_size = len(vocab.word2idx)

    if arch.lower() == "rnn":
        model = RNNClassifier(vocab_size=vocab_size).to(device)
    else:
        model = LSTMClassifier(vocab_size=vocab_size).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": []}
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for x_b, y_b in train_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            logits = model(x_b)
            loss = criterion(logits, y_b)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)

        model.eval()
        v_loss = 0.0
        with torch.no_grad():
            for x_b, y_b in val_loader:
                x_b, y_b = x_b.to(device), y_b.to(device)
                logits = model(x_b)
                v_loss += criterion(logits, y_b).item()
        val_loss = v_loss / len(val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        print(f"Epoch {epoch}/{epochs} - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    train_latency = (time.time() - start_time) / epochs

    # Test evaluation
    model.eval()
    all_preds, all_probs = [], []
    eval_start = time.time()
    with torch.no_grad():
        for x_b, _ in test_loader:
            x_b = x_b.to(device)
            logits = model(x_b)
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(probs.argmax(axis=-1))

    inference_ms = ((time.time() - eval_start) / len(test_df)) * 1000

    probs_arr = np.array(all_probs)
    metrics = compute_metrics(test_df["label"].values, all_preds, probs_arr[:, 1])
    print(f"\n[{name}] Accuracy: {metrics['accuracy']:.4f} | F1: {metrics['f1']:.4f}")

    out_dir = MODELS_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), out_dir / "model.pt")
    vocab.save(out_dir / "vocab.json")

    meta = {
        "name": name,
        "arch": arch,
        "base_model": f"PyTorch {arch.upper()}",
        "max_len": max_len,
        "sample": sample,
        "epochs": epochs,
        "train_size": len(train_df),
        "test_metrics": {k: metrics[k] for k in ("accuracy", "precision", "recall", "f1")},
        "inference_ms": round(inference_ms, 2)
    }
    (out_dir / "meta.json").write_text(json.dumps(meta, indent=2))
    save_results(name, metrics, history)
    print(f"[{name}] Model & metrics saved to {out_dir}\n")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--arch", choices=["rnn", "lstm"], default="lstm")
    parser.add_argument("--name", type=str, default=None)
    parser.add_argument("--sample", type=int, default=3000)
    parser.add_argument("--epochs", type=int, default=3)
    args = parser.parse_args()

    train_model(arch=args.arch, name=args.name, sample=args.sample, epochs=args.epochs)

In [ ]:
%%writefile src/train_baseline.py
"""Baseline: TF-IDF + Logistic Regression.

    python -m src.train_baseline            # full dataset (fast, ~1 min on CPU)
    python -m src.train_baseline --sample 3000
"""
from __future__ import annotations

import argparse
import time

import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from .config import MODELS_DIR, SEED
from .data import get_splits
from .metrics import compute_metrics, save_results


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--sample", type=int, default=None)
    p.add_argument("--name", default="tfidf_logreg")
    args = p.parse_args()

    train_df, val_df, test_df = get_splits(sample=args.sample)
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2),
                                  min_df=2, max_df=0.9, max_features=200_000, sublinear_tf=True)),
        ("clf", LogisticRegression(C=4.0, max_iter=2000, random_state=SEED)),
    ])
    t0 = time.time()
    pipe.fit(train_df["content"], train_df["label"])
    train_time = time.time() - t0

    probs = pipe.predict_proba(test_df["content"])
    metrics = compute_metrics(test_df["label"].values, probs.argmax(1), probs[:, 1])
    save_results(args.name, metrics, {"base_model": "TF-IDF (1-2 gram) + Logistic Regression",
                                      "train_size": len(train_df), "train_time_sec": round(train_time, 1)})
    MODELS_DIR.mkdir(exist_ok=True)
    joblib.dump(pipe, MODELS_DIR / f"{args.name}.joblib")
    print(f"[baseline] saved -> {MODELS_DIR / (args.name + '.joblib')}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/compare.py
"""Collect every reports/*_metrics.json into one comparison table + bar chart.

    python -m src.compare
"""
from __future__ import annotations

import json

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from .config import FIGURES_DIR, REPORTS_DIR

PRETTY = {
    "tfidf_logreg": "TF-IDF + LogReg",
    "rnn": "Simple RNN",
    "lstm": "BiLSTM",
    "bert": "BERT",
    "roberta": "RoBERTa",
    "distilbert": "DistilBERT"
}
ORDER = ["tfidf_logreg", "rnn", "lstm", "distilbert", "bert", "roberta"]


def load_all() -> pd.DataFrame:
    rows = []
    for f in REPORTS_DIR.glob("*_metrics.json"):
        m = json.loads(f.read_text())
        rows.append({
            "key": m["model"], "Model": PRETTY.get(m["model"], m["model"]),
            "Accuracy": m["accuracy"], "Precision": m["precision"], "Recall": m["recall"],
            "F1 Score": m["f1"], "ROC-AUC": m.get("roc_auc"),
            "Train size": m.get("train_size"), "Test size": m.get("n_test"),
            "Train time (s)": m.get("train_time_sec"),
        })
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df["_o"] = df["key"].map(lambda k: ORDER.index(k) if k in ORDER else 99)
    return df.sort_values("_o").drop(columns="_o").reset_index(drop=True)


def main():
    df = load_all()
    if df.empty:
        print("No metrics found. Train some models first.")
        return
    cols = ["Accuracy", "Precision", "Recall", "F1 Score"]
    table = df[["Model", *cols, "ROC-AUC", "Train size", "Test size", "Train time (s)"]]
    print(table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    md = table.copy()
    for c in cols + ["ROC-AUC"]:
        md[c] = md[c].map(lambda v: f"{v * 100:.2f}%" if pd.notna(v) else "—")
    lines = ["| " + " | ".join(md.columns) + " |", "|" + "---|" * len(md.columns)]
    lines += ["| " + " | ".join("—" if pd.isna(v) else str(v) for v in row) + " |" for row in md.values]
    (REPORTS_DIR / "comparison.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    table.to_csv(REPORTS_DIR / "comparison.csv", index=False)

    x = np.arange(len(cols))
    w = 0.8 / len(df)
    palette = ["#8b8fa3", "#ffb86b", "#7c5cff", "#00d4a6", "#ff5c8a"]
    fig, ax = plt.subplots(figsize=(8, 4.2), dpi=130)
    for i, r in df.iterrows():
        bars = ax.bar(x + i * w - 0.4 + w / 2, [r[c] for c in cols], w, label=r["Model"],
                      color=palette[i % len(palette)])
        ax.bar_label(bars, fmt="%.3f", fontsize=7, padding=2)
    lo = max(0.0, df[cols].values.min() - 0.08)
    ax.set_ylim(lo, 1.0)
    ax.set_xticks(x, cols)
    ax.set_ylabel("score (test set)")
    ax.set_title("Model Comparison — Fake News Detection", weight="bold")
    ax.legend(ncol=len(df), loc="upper center", bbox_to_anchor=(0.5, -0.1), frameon=False)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "model_comparison.png", facecolor="white")
    print(f"\nSaved {REPORTS_DIR / 'comparison.csv'} and figures/model_comparison.png")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/predict.py
"""Inference + explainability for all trained models.

Explainability
--------------
* Transformers (BERT / RoBERTa): **Integrated Gradients** on the word embeddings.
  We integrate the gradient of (logit_FAKE - logit_REAL) along a straight path
  from a "blank" input ([PAD] embeddings) to the real input. The result is a
  per-token score that tells how much each token pushed the prediction towards
  FAKE (positive) or towards REAL (negative). Token scores are summed per word.
* TF-IDF + Logistic Regression: word contribution = tf-idf weight x coefficient.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch

from .config import LABELS, MODELS_DIR, REPORTS_DIR
from .data import clean_text

_WORD_RE = re.compile(r"\S+")


@dataclass
class Prediction:
    label: str                      # "FAKE" | "REAL"
    confidence: float               # probability of the predicted class
    probs: dict                     # {"FAKE": p, "REAL": p}
    words: list = field(default_factory=list)   # [(word, score)] score>0 -> FAKE
    n_tokens: int = 0
    truncated: bool = False


def _words_with_spans(text):
    return [(m.group(), m.start(), m.end()) for m in _WORD_RE.finditer(text)]


def _normalise(scores: np.ndarray) -> np.ndarray:
    m = np.abs(scores).max() if len(scores) else 0
    return scores / m if m > 0 else scores


# =========================================================================
class TransformerPredictor:
    kind = "transformer"

    def __init__(self, model_dir: Path, device: str | None = None):
        from .model import FakeNewsClassifier

        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model, self.tokenizer, self.meta = FakeNewsClassifier.load(model_dir, self.device)
        self.name = self.meta.get("name", Path(model_dir).name)
        self.max_len = int(self.meta.get("max_len", 256))

    def tokenize(self, text: str):
        return self.tokenizer(clean_text(text), truncation=True, max_length=self.max_len,
                              return_offsets_mapping=True, return_tensors="pt")

    @torch.no_grad()
    def predict_proba(self, texts: list[str]) -> np.ndarray:
        enc = self.tokenizer([clean_text(t) for t in texts], truncation=True, max_length=self.max_len,
                             padding=True, return_tensors="pt").to(self.device)
        logits = self.model(enc["input_ids"], enc["attention_mask"])
        return torch.softmax(logits, -1).cpu().numpy()

    def predict(self, text: str, explain: bool = False, ig_steps: int = 24) -> Prediction:
        p = self.predict_proba([text])[0]
        idx = int(p.argmax())
        pred = Prediction(LABELS[idx], float(p[idx]), {"FAKE": float(p[0]), "REAL": float(p[1])})
        full_len = len(self.tokenizer(clean_text(text))["input_ids"])
        pred.n_tokens, pred.truncated = min(full_len, self.max_len), full_len > self.max_len
        if explain:
            pred.words = self.explain(text, ig_steps)
        return pred

    # ---------------------------------------------------- integrated grads --
    def explain(self, text: str, steps: int = 24, chunk: int = 8):
        text = clean_text(text)
        enc = self.tokenize(text)
        offsets = enc.pop("offset_mapping")[0].tolist()
        ids = enc["input_ids"].to(self.device)
        mask = enc["attention_mask"].to(self.device)

        special = torch.tensor(self.tokenizer.get_special_tokens_mask(
            ids[0].tolist(), already_has_special_tokens=True), device=self.device).bool()
        pad_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else 0
        base_ids = torch.where(special, ids[0], torch.full_like(ids[0], pad_id)).unsqueeze(0)

        emb = self.model.encoder.get_input_embeddings()
        with torch.no_grad():
            x, x0 = emb(ids), emb(base_ids)           # (1, L, H)
        delta = x - x0
        alphas = (torch.arange(1, steps + 1, device=self.device).float() - 0.5) / steps  # midpoint rule
        total = torch.zeros_like(x[0])

        self.model.zero_grad(set_to_none=True)
        for a in alphas.split(chunk):
            path = (x0 + a.view(-1, 1, 1) * delta).detach().requires_grad_(True)
            logits = self.model(inputs_embeds=path, attention_mask=mask.expand(len(a), -1))
            target = (logits[:, 0] - logits[:, 1]).sum()      # towards FAKE
            (grad,) = torch.autograd.grad(target, path)
            total += grad.sum(0)
        token_attr = ((total / steps) * delta[0]).sum(-1).detach().cpu().numpy()
        token_attr[special.cpu().numpy()] = 0.0

        # aggregate token scores into whitespace-separated words
        words = _words_with_spans(text)
        scores = np.zeros(len(words))
        wi = 0
        for (s, e), sc in zip(offsets, token_attr):
            if e <= s:
                continue
            while wi < len(words) and words[wi][2] <= s:
                wi += 1
            if wi < len(words) and words[wi][1] <= s < words[wi][2]:
                scores[wi] += sc
        last_char = max((e for s, e in offsets), default=0)
        covered = [w[1] < last_char for w in words]
        scores = _normalise(scores)
        return [(w[0], float(sc) if cov else None) for w, sc, cov in zip(words, scores, covered)]


# =========================================================================
class BaselinePredictor:
    kind = "baseline"

    def __init__(self, path: Path):
        import joblib

        self.pipe = joblib.load(path)
        self.name = Path(path).stem
        self.meta = {"name": self.name, "base_model": "TF-IDF + Logistic Regression"}
        mpath = REPORTS_DIR / f"{self.name}_metrics.json"
        if mpath.exists():
            m = json.loads(mpath.read_text())
            self.meta["test_metrics"] = {k: m[k] for k in ("accuracy", "precision", "recall", "f1")}
            self.meta["train_size"] = m.get("train_size")
        tfidf, clf = self.pipe.named_steps["tfidf"], self.pipe.named_steps["clf"]
        self.vocab, self.coef = tfidf.vocabulary_, clf.coef_[0]   # coef > 0 -> REAL

    def predict_proba(self, texts):
        return self.pipe.predict_proba([clean_text(t) for t in texts])

    def predict(self, text: str, explain: bool = False, **_) -> Prediction:
        p = self.predict_proba([text])[0]
        idx = int(p.argmax())
        pred = Prediction(LABELS[idx], float(p[idx]), {"FAKE": float(p[0]), "REAL": float(p[1])})
        pred.n_tokens = len(text.split())
        if explain:
            pred.words = self.explain(text)
        return pred

    def explain(self, text: str):
        text = clean_text(text)
        tfidf = self.pipe.named_steps["tfidf"]
        vec = tfidf.transform([text])
        weight = {i: v for i, v in zip(vec.indices, vec.data)}
        words = _words_with_spans(text)
        scores = []
        for w, _, _ in words:
            tok = re.sub(r"[^\w']", "", w.lower())
            j = self.vocab.get(tok)
            scores.append(-self.coef[j] * weight.get(j, 0.0) if j is not None else 0.0)
        scores = _normalise(np.array(scores, dtype=float))
        return [(w[0], float(s)) for w, s in zip(words, scores)]


# =========================================================================
def available_models() -> dict[str, Path]:
    """Discover trained models in models/. Transformers, RNN/LSTM, and Baselines."""
    found = {}
    for d in sorted(MODELS_DIR.iterdir()) if MODELS_DIR.exists() else []:
        if d.is_dir() and (d / "meta.json").exists():
            if (d / "model.pt").exists() or (d / "vocab.json").exists():
                found[d.name] = d
    for f in sorted(MODELS_DIR.glob("*.joblib")):
        found[f.stem] = f
    order = ["bert", "roberta", "distilbert", "lstm", "rnn", "tfidf_logreg"]
    return dict(sorted(found.items(), key=lambda kv: order.index(kv[0]) if kv[0] in order else 50))


def load_predictor(path: Path):
    path = Path(path)
    if path.suffix == ".joblib":
        return BaselinePredictor(path)
    meta_path = path / "meta.json"
    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
        if meta.get("arch") in ("rnn", "lstm"):
            from .models_rnn_lstm import RNNLSTMPredictor
            return RNNLSTMPredictor(path)
    return TransformerPredictor(path)


if __name__ == "__main__":
    import sys

    models = available_models()
    if not models:
        sys.exit("No trained models found in models/.")
    name = sys.argv[1] if len(sys.argv) > 1 and sys.argv[1] in models else next(iter(models))
    pr = load_predictor(models[name])
    text = " ".join(a for a in sys.argv[1:] if a != name) or input("News text: ")
    r = pr.predict(text, explain=True)
    print(f"\nModel: {name}\nPrediction: {r.label}\nConfidence: {r.confidence * 100:.1f}%")
    top = sorted([w for w in r.words if w[1] is not None], key=lambda w: -abs(w[1]))[:8]
    print("Most influential words:", ", ".join(f"{w} ({'FAKE' if s > 0 else 'REAL'})" for w, s in top))

## 2 · Dataset preparation & quick EDA

In [ ]:
!python -m src.data

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.data import get_splits
train, val, test = get_splits()
df = pd.concat([train, val, test])
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x=df.label.map({0: 'Fake', 1: 'Real'}), ax=ax[0], palette=['#ff4d6d', '#00d4a6'])
ax[0].set_title('Class distribution'); ax[0].set_xlabel('')
df['words'] = df.content.str.split().str.len()
sns.histplot(data=df, x='words', hue=df.label.map({0: 'Fake', 1: 'Real'}), bins=80, ax=ax[1],
             palette=['#ff4d6d', '#00d4a6'], element='step')
ax[1].set_xlim(0, 1500); ax[1].set_title('Article length (words)')
plt.tight_layout(); plt.show()
print(f"train={len(train):,}  val={len(val):,}  test={len(test):,}")

## 3 · Baseline Models — TF-IDF, Simple RNN & BiLSTM

In [ ]:
!python -m src.train_baseline

In [ ]:
!python -m src.train_rnn_lstm --arch rnn --epochs 3

In [ ]:
!python -m src.train_rnn_lstm --arch lstm --epochs 3

## 4 · Fine-tune BERT (`bert-base-uncased`)

In [ ]:
!python -m src.train_transformer --model bert --epochs 2 --max-len 256 --batch-size 16 --log-every 50

## 5 · Fine-tune RoBERTa (`roberta-base`)

In [ ]:
!python -m src.train_transformer --model roberta --epochs 2 --max-len 256 --batch-size 16 --log-every 50

## 6 · Comprehensive Model Comparison (TF-IDF vs Simple RNN vs BiLSTM vs BERT vs RoBERTa)

In [ ]:
!python -m src.compare
from IPython.display import Image, display, Markdown
display(Markdown(open('reports/comparison.md').read()))
display(Image('reports/figures/model_comparison.png'))
for n in ['tfidf_logreg', 'rnn', 'lstm', 'bert', 'roberta']:
    p = f'reports/figures/{n}_confusion_matrix.png'
    if os.path.exists(p): display(Image(p, width=380))

## 7 · Source Analysis & Fact-Check Validation Test (English & Hindi)

In [ ]:
from src.predict import available_models, load_predictor
from src.source_verifier import analyze_source

models = available_models()
predictor = load_predictor(models['bert']) if 'bert' in models else load_predictor(next(iter(models.values())))

test_cases = [
    ("Scientists discover a new planet that can support human life.", "nasa.gov"),
    ("Government announces that humans can now live without food.", "fake-news.co"),
    ("ISRO launches new earth observation satellite successfully.", "pib.gov.in"),
    ("सरकारी आयोग का ऐलान: इंसानों को भोजन की जरूरत नहीं होगी!", "unverified-blog.com")
]

for text, src in test_cases:
    r = predictor.predict(text, explain=True)
    src_res = analyze_source(src, r.label, r.confidence)
    print(f"Text: {text}\nSource: {src_res.domain} (Credibility: {src_res.credibility})")
    print(f"Prediction: {r.label} | Confidence: {r.confidence*100:.1f}% | Combined Trust Index: {src_res.combined_confidence*100:.1f}%\n" + "-"*60)

## 8 · Download trained models for local Streamlit app

In [ ]:
!zip -qr trained_models.zip models reports
from google.colab import files
files.download('trained_models.zip')